# §63 / §63.2 — DoTA train captions: **pilot on 30 clips**, prompt **v1 vs v2** (Qwen2.5-VL)

Rule: `core/docs/final/KIM_RUN.md` **§63**, **§63.2** (prompt v2). Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **A data artifact, not an arm**: nothing trains, and no caption is used anywhere until you decide.

What it does: 30 recovered DoTA train clips (one per accident type + 20 at random, seed 2024) → 16 frames over `[onset − 2 s, min(onset + 3 s, anomaly_end)]` at 700 × 392 → Qwen2.5-VL with a dashcam prompt that carries DoTA's own label → one sentence per clip, **once per prompt version, one model load**:

* **v1** = §63's prompt (unchanged; on the L4 its 32B captions already exist and are skipped).
* **v2** = one template per ego flag (§63.2): non-ego clips are told to find the moment of impact, ignore road users that only pass the camera, never name an unseen one, and state only what is certain; both templates ask for the exact class, a colour only when clear, and no "the vehicles stop" unless visible.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | mount, paths (**model picked from the GPU**), annotation zip, deps | no | 3-4 min |
| 4 | preflight + both prompts printed for one ego and one non-ego clip | no | 1 min |
| 5 | pytest — **raises if red** | no | < 1 min |
| 6 | pin the model revision | no | seconds |
| 7 | **caption the 30 clips with v1 and v2** (resumable) | yes | L4 32B: ~15 min (v1 already done) + 4 min load; A100-80GB 72B: ~10 min download + ~35 min |
| 8 | v1 vs v2 side by side + the 30 contact sheets | no | 1 min |
| 9 | ratings → `qc-score` per version → **paste to Claude** | no | your time |

**Model by GPU (automatic in step 2; each model and prompt writes its own folder, records are never mixed):**

| GPU | model | quant | VRAM |
|---|---|---|---|
| A100-80GB / H100 | `Qwen/Qwen2.5-VL-72B-Instruct` | nf4 | ~45 GB |
| A100-40GB, L4 (24 GB) | `Qwen/Qwen2.5-VL-32B-Instruct` | nf4 | ~20 GB |
| L4 if the 32B runs out of memory | set `MODEL_OVERRIDE = 'Qwen/Qwen2.5-VL-7B-Instruct'` | none (bf16) | ~17 GB |

**Compare v1 and v2 on the same model only.** On the A100 both run on 72B, so that pair is clean; never put the L4's 32B v1 beside the A100's 72B v2. Thesis is only read; everything new goes to `Thesis-Final/outputs/dota_captions/`.


In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 2. Paths, code, annotation zip

In [2]:
import hashlib
import json
import os
import subprocess
import sys
import time
import urllib.request
from collections import Counter
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, DoTA train frames, every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'
os.environ['HF_HOME'] = '/content/hf'             # model weights on VM disk, not Drive FUSE

REPO = Path(f'{DRIVE_F}/kat-vad')
assert (REPO / 'core/tools/dota_caption.py').is_file(), 'this upload has no core/tools/dota_caption.py -- re-upload kat-vad/'
assert '### 63.2 ' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §63.2 (prompt v2) -- re-upload kat-vad/'
assert '## 63. ' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §63 -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'DOTA_CAPTION_PROMPT_V2'), 'constants has no prompt v2 -- re-upload kat-vad/'
import torch  # noqa: E402

# MODEL by GPU (table in the first cell): 72B on >= 70 GB, else 32B; 7B only by hand after a 32B OOM
MODEL_OVERRIDE = None                              # e.g. 'Qwen/Qwen2.5-VL-7B-Instruct' after an OOM on the L4
PROMPTS = list(constants.DOTA_CAPTION_PROMPT_NAMES)   # ['v1', 'v2']: v1 is skipped clip by clip where it exists
GPU_GB  = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0.0
MODEL   = MODEL_OVERRIDE or ('Qwen/Qwen2.5-VL-72B-Instruct' if GPU_GB >= 70 else 'Qwen/Qwen2.5-VL-32B-Instruct')
print(f'GPU {torch.cuda.get_device_name(0) if GPU_GB else "NONE"} {GPU_GB:.0f} GB -> {MODEL} | prompts {PROMPTS}')
# NEEDS[size] = (quant, VRAM GB, download GB); the 7B runs unquantized (fits a 24 GB L4 in bf16)
NEEDS   = {'72B': (constants.NEXAR_CAPTION_QUANT_NF4, 70, 150), '32B': (constants.NEXAR_CAPTION_QUANT_NF4, 21, 70),
           '7B': (constants.NEXAR_CAPTION_QUANT_NONE, 21, 20)}
SIZE    = next(k for k in NEEDS if f'-{k}-' in MODEL)
QUANT, NEED_GB, WEIGHTS_GB = NEEDS[SIZE]
print(f'model {MODEL} ({SIZE}, quant {QUANT})')
ARCHIVE = Path(DRIVE_F) / 'data' / constants.DOTA_TRAIN_FRAMES_DIRNAME     # {dota_id}.zip (§52)
OUT     = constants.OUTPUT_ROOT / constants.DOTA_CAPTION_OUTPUT_DIRNAME
NEXAR_PIN = constants.OUTPUT_ROOT / constants.NEXAR_CAPTION_OUTPUT_DIRNAME / f"revision_{MODEL.split('/')[-1]}.json"
LOCAL   = Path('/content/dota_captions')
for p in (OUT, LOCAL):
    p.mkdir(parents=True, exist_ok=True)

ANN = LOCAL / 'DoTA_annotations.zip'              # DoTA's official annotation zip (10 MB)
if not ANN.exists():
    urllib.request.urlretrieve('https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip', ANN)
frozen = json.loads((constants.DOTA_TRAIN_SPLITS_DIR / constants.DOTA_TRAIN_MAP_FILENAME).read_text())
assert hashlib.sha256(ANN.read_bytes()).hexdigest() == frozen['sources']['annotations_sha256'], 'annotation zip is not the one the map was built on'
print('paths OK |', MODEL, QUANT, '| out', OUT)

GPU NVIDIA L4 22 GB -> Qwen/Qwen2.5-VL-32B-Instruct | prompts ['v1', 'v2']
model Qwen/Qwen2.5-VL-32B-Instruct (32B, quant nf4)
paths OK | Qwen/Qwen2.5-VL-32B-Instruct nf4 | out /content/drive/MyDrive/Thesis-Final/outputs/dota_captions


## 3. Dependencies

In [3]:
%%bash
pip install -q "transformers==4.56.*" av "bitsandbytes>=0.45"
python -c "import av, bitsandbytes, torch, transformers; print('deps OK', transformers.__version__, 'bnb', bitsandbytes.__version__, 'torch', torch.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 7 needs one'
df -h /content | tail -1

deps OK 4.56.2 bnb 0.50.2 torch 2.14.1+cu130
NVIDIA L4, 23034 MiB
overlay         236G  615M  236G   1% /


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 4. Preflight (CPU)

In [4]:
import shutil  # noqa: E402

import torch  # noqa: E402

from core.tools import dota_caption as dc  # noqa: E402
from core.tools import nexar_caption as nc  # noqa: E402
from core.tools.dota_train_extract import train_clips  # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')


def run(cmd):
    '''Stream a child's output here; raise with its tail on failure.'''
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


CLIPS = train_clips(constants.DOTA_TRAIN_SPLITS_DIR / constants.DOTA_TRAIN_MAP_FILENAME)
assert len(CLIPS) == 3020, len(CLIPS)
ANNS = dc.load_annotations(ANN, sorted(CLIPS))                  # raises on an unknown accident type
for vid, a in ANNS.items():
    dc.caption_window(a)                                         # raises if a clip cannot give 16 frames
PILOT = dc.pilot_sample(ANNS)
missing = [v for v in PILOT if not (ARCHIVE / f'{v}.zip').is_file()]
assert not missing, f'{len(missing)} pilot archives missing under {ARCHIVE}: {missing[:5]}'
print(f'preflight OK: {len(CLIPS)} clips, {len(PILOT)} pilot archives on Drive')
print('pilot types:', dict(Counter(ANNS[v]['accident_name'] for v in PILOT)))
print('pilot ego-involved:', sum(ANNS[v]['ego_involve'] for v in PILOT), '/', len(PILOT))
EGO_EX = next(v for v in PILOT if ANNS[v]['ego_involve'])
NON_EX = next(v for v in PILOT if not ANNS[v]['ego_involve'])
for version in PROMPTS:
    for vid in (EGO_EX, NON_EX):
        print(f"\n[{version}, ego={ANNS[vid]['ego_involve']}] {dc.prompt_for(ANNS[vid], version)}")
DIRS = {v: OUT / dc.PILOT_DIRNAME / dc.pilot_dirname(MODEL, v) for v in PROMPTS}
for v, d in DIRS.items():
    done = len(list((d / nc.CAPTIONS_DIRNAME).glob('*.json'))) if d.exists() else 0
    print(f'{v}: {d.name} -- {done} / {len(PILOT)} already captioned')

preflight OK: 3020 clips, 30 pilot archives on Drive
pilot types: {'turning': 9, 'leave_to_left': 1, 'start_stop_or_stationary': 2, 'oncoming': 5, 'obstacle': 2, 'moving_ahead_or_waiting': 4, 'lateral': 4, 'unknown': 1, 'leave_to_right': 1, 'pedestrian': 1}
pilot ego-involved: 19 / 30

[v1, ego=True] This video was recorded by a dashcam mounted behind the windshield of the ego vehicle, facing forward. It shows a traffic accident. The annotators labelled it as a collision with a vehicle that turns into or crosses the road. The ego vehicle (the camera car) is one of the vehicles in the accident. Describe the accident in one sentence of at most 40 words: which road users collide (cars, trucks, buses, motorcycles, cyclists, pedestrians, the ego vehicle), what each one does just before the impact, from which direction it comes, and what happens right after. Name only road users that take part. Use the label as a hint, but describe what is visible; if the video disagrees with the label, desc

## 5. Tests (CPU) — **raises if anything fails: stop and paste the output**

In [5]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_dota_caption.py', 'core/tests/test_nexar_caption.py'])

................................................                         [100%]
48 passed in 20.23s


## 6. Pin the model revision once (C4)

If Nexar's captions already pinned this model, the same commit is reused (same weights as §29.3); otherwise the Hub's current commit is written to `OUT/revision_<model>.json`.

In [6]:
from huggingface_hub import HfApi  # noqa: E402

REV_FILE = OUT / f"revision_{MODEL.split('/')[-1]}.json"
if not REV_FILE.exists():
    pin = json.loads(NEXAR_PIN.read_text()) if NEXAR_PIN.exists() else {'model': MODEL, 'revision': HfApi().model_info(MODEL).sha}
    REV_FILE.write_text(json.dumps(pin, indent=2))
REV = json.loads(REV_FILE.read_text())
assert REV['model'] == MODEL, REV
REVISION = REV['revision']
print(f'{MODEL} @ {REVISION}', '(from Nexar pin)' if NEXAR_PIN.exists() else '')

Qwen/Qwen2.5-VL-32B-Instruct @ 7cfb30d71a1f4f49a57592323337a4a4727301da 


## 7. Pilot: caption the 30 clips (GPU, resumable)

A disconnect is harmless: re-run steps 1-6 and this cell; finished clips are skipped.

In [7]:
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'needs a bf16 GPU (L4 / A100 / H100)'
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
assert gb >= NEED_GB, f'{gb:.0f} GB GPU: {MODEL} needs >= {NEED_GB} GB -- pick a smaller MODEL in step 2'
free = shutil.disk_usage('/content').free / 2**30
cached = Path(os.environ['HF_HOME']) / 'hub' / f"models--{MODEL.replace('/', '--')}"
assert cached.exists() or free > WEIGHTS_GB + 10, f'free {free:.0f} GB on /content; the weights need ~{WEIGHTS_GB} GB'
ENV['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'   # less fragmentation on a 24 GB card
print(f'GPU {torch.cuda.get_device_name(0)} {gb:.0f} GB | disk free {free:.0f} GB')
t0 = time.time()
try:
    run([sys.executable, '-m', 'core.tools.dota_caption', 'pilot', '--archive-dir', ARCHIVE, '--annotations', ANN,
         '--out-dir', OUT, '--revision', REVISION, '--model', MODEL, '--quant', QUANT, '--prompts', *PROMPTS])
except RuntimeError:
    if SIZE != '7B':
        print(f"\nIf the tail above says 'CUDA out of memory': set MODEL_OVERRIDE = 'Qwen/Qwen2.5-VL-7B-Instruct' in step 2, "
              f"re-run steps 2, 6 and 7 (the {SIZE} captions already written are kept in their own folder).")
    raise
print(f'done in {(time.time() - t0) / 60:.0f} min')
KEYS = ('prompt_tag', 'clips', 'distinct', 'words_median', 'words_max', 'over_max_words', 'non_ego_clips',
        'non_ego_naming_ego', 'scene_word_share')
for v, d in DIRS.items():
    summary = json.loads((d / dc.PILOT_SUMMARY_FILENAME).read_text())
    assert summary['clips'] == len(PILOT) and summary['ids'] == PILOT and summary['revision'] == REVISION, summary
    print(v, {k: summary.get(k) for k in KEYS})

GPU NVIDIA L4 22 GB | disk free 235 GB
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!

Fetching 18 files: 100%|██████████| 18/18 [03:09<00:00, 10.51s/it]

Loading checkpoint shards: 100%|██████████| 18/18 [04:00<00:00, 13.35s/it]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
done in 16 min
v1 {'prompt_tag': '7eb9f067', 'clips': 30, 'distinct': 30, 'words_median': 27.0, 'words_max': 37, 'over_max_words': 0, 'non_ego_clips': 11, 'non_ego_naming_ego': 0, 'scene_word_share': 0.8333333333333334}
v2 {'prompt_tag': 'a1e1

## 8. v1 vs v2 side by side + the 16 frames the model saw

On the L4 (32B) the v1 column carries Claude's §63.1 rating in **[ ]** once you run the fill cell below; v2 is unrated. **How to rate v2** (Tab. B 1–5, judged against the frames, not against the label): 5 = accurate and complete; 4 = minor slips; 3 = main event right but notable omissions / errors; 2 = largely wrong; 1 = nothing to do with the video. Optional `type_ok` / `roles_ok` = y / n.

In [8]:
# §63.1: Claude's ratings of the 32B v1 captions (by rank), written into that folder's qc_ratings.csv.
# They belong to the 32B v1 folder only: skipped on any other model.
import csv  # noqa: E402

V1_CLAUDE = {1: (4, 'y', 'y'), 2: (4, 'y', 'y'), 3: (4, 'y', 'y'), 4: (3, 'y', 'y'), 5: (3, 'y', 'y'),
             6: (3, 'y', 'n'), 7: (4, 'y', 'n'), 8: (4, 'y', 'y'), 9: (5, 'y', 'y'), 10: (2, 'n', 'n'),
             11: (4, 'y', 'y'), 12: (3, 'y', 'y'), 13: (3, 'y', 'n'), 14: (3, 'y', 'n'), 15: (4, 'y', 'y'),
             16: (3, 'n', 'n'), 17: (4, 'y', 'y'), 18: (5, 'y', 'y'), 19: (4, 'y', 'y'), 20: (3, 'y', 'n'),
             21: (2, 'y', 'n'), 22: (5, 'y', 'y'), 23: (5, 'y', 'y'), 24: (1, 'n', 'n'), 25: (4, 'y', 'y'),
             26: (4, 'y', 'y'), 27: (4, 'y', 'y'), 28: (2, 'y', 'n'), 29: (4, 'y', 'y'), 30: (4, 'y', 'y')}
if '32B' in MODEL and 'v1' in DIRS:
    path = DIRS['v1'] / nc.QC_DIRNAME / nc.QC_RATINGS_FILENAME
    rows = list(csv.DictReader(path.open(newline='', encoding='utf-8')))
    assert len(rows) == 30 and {int(r['rank']) for r in rows} == set(V1_CLAUDE)
    if not any(r['score'] for r in rows):                   # never overwrite scores already given
        for r in rows:
            r['score'], r['type_ok'], r['roles_ok'] = map(str, V1_CLAUDE[int(r['rank'])])
        with path.open('w', newline='', encoding='utf-8') as fh:
            w = csv.DictWriter(fh, fieldnames=list(rows[0]))
            w.writeheader()
            w.writerows(rows)
        print('v1 (32B): Claude ratings written to', path)
    else:
        print('v1 (32B): scores already present -- left as they are')
else:
    print(f'{MODEL}: no Claude ratings for this model -- rate both columns yourself')

v1 (32B): Claude ratings written to /content/drive/MyDrive/Thesis-Final/outputs/dota_captions/pilot/7eb9f067_Qwen2.5-VL-32B-Instruct_16f/qc/qc_ratings.csv


In [ ]:
from IPython.display import Image, Markdown, display  # noqa: E402

CMP = OUT / dc.PILOT_DIRNAME / f"compare_{MODEL.split('/')[-1]}_{'_'.join(PROMPTS)}.md"
run([sys.executable, '-m', 'core.tools.dota_caption', 'compare', *[x for v, d in DIRS.items() for x in ('--pilot', v, d)],
     '--out', CMP])
display(Markdown(CMP.read_text()))
QC = DIRS[PROMPTS[-1]] / nc.QC_DIRNAME                       # same frames for every version
for png in sorted(QC.glob('*.png')):
    print(png.stem)
    display(Image(filename=str(png), width=900))
print(f"Rate v2 in {QC / nc.QC_RATINGS_FILENAME} on Drive (score 1-5, optional type_ok / roles_ok), save, then run step 9.")

## 9. QC score per version — **after rating v2 in its `qc_ratings.csv`; paste the printed block to Claude**

A version with no score yet is reported and skipped.

In [ ]:
for v, d in DIRS.items():
    rows = list(csv.DictReader((d / nc.QC_DIRNAME / nc.QC_RATINGS_FILENAME).open(newline='', encoding='utf-8')))
    if not all(r['score'].strip() for r in rows):
        print(f'{v}: {sum(not r["score"].strip() for r in rows)} rows without a score -- not scored')
        continue
    run([sys.executable, '-m', 'core.tools.dota_caption', 'qc-score', '--pilot-dir', d])
    print(v, json.dumps(json.loads((d / nc.QC_DIRNAME / nc.QC_SUMMARY_FILENAME).read_text()), indent=1))
run([sys.executable, '-m', 'core.tools.dota_caption', 'compare', *[x for v, d in DIRS.items() for x in ('--pilot', v, d)],
     '--out', CMP])
print(CMP.read_text())
drive.flush_and_unmount()          # C10: every pending Drive write lands before the VM goes